In [33]:
import argparse
import torch
import random
import sys
import gc
from pathlib import Path
from tqdm import tqdm
from collections import Counter
from datasets import load_dataset, concatenate_datasets
from span_marker import SpanMarkerModel
from gliner import GLiNER

# Import path resolution logic
from EXPERIMENTS.finetune import get_output_dir
# from dataset_processing import shorten_name

# --- CONFIGURATION ---
DEFAULT_SEEDS = [0, 42, 3012, 33, 131]

all_tags = """Asset
Body of Water
Body Part
Chemical
Disease
Ecosystem
Energy Source
Field of Study
Geographical Feature
Intellectual Artefact
Location
Mathematical Expression
Measuring Device
Meteorological Phenomenon
Method
Natural Disaster
Natural Phenomenon
Organism
Organization
Other
Person
Physical Artefact
Physical Phenomenon
Policy or Objective
Quantity
Satellite
System
Time Period"""

TARGET_TAGS = all_tags.split("\n")

# --- THRESHOLDS (The 4-1 Rule) ---
SUCCESS_THRESHOLD = 3  # Must be found in >= 4 seeds
FAILURE_THRESHOLD = 2  # Must be found in <= 1 seed

def load_gold_data(dataset_id):
    print(f"--- Loading Dataset: {dataset_id} ---")
    ds = load_dataset(dataset_id)
    splits = [ds[split] for split in ds.keys()]
    merged = concatenate_datasets(splits)
    features = merged.features["ner_tags"].feature.names
    id2label = {i: name for i, name in enumerate(features)}
    return merged, id2label

def extract_gold_spans(row, id2label):
    tokens = row['tokens']
    tags = row['ner_tags']
    extracted_gold = []
    curr_start, curr_label = None, None
    for i, tag_id in enumerate(tags):
        tag_name = id2label[tag_id]
        if tag_name.startswith("B-"):
            if curr_label: extracted_gold.append((" ".join(tokens[curr_start:i]), curr_label))
            curr_start, curr_label = i, tag_name[2:]
        elif tag_name.startswith("I-") and curr_label == tag_name[2:]: continue
        else:
            if curr_label: extracted_gold.append((" ".join(tokens[curr_start:i]), curr_label))
            curr_label, curr_start = None, None
    if curr_label: extracted_gold.append((" ".join(tokens[curr_start:]), curr_label))
    return set(extracted_gold)

def run_inference_single_seed(model_path, model_type, texts, device):
    predictions = []
    if model_type == "SPANMARKER":
        model = SpanMarkerModel.from_pretrained(model_path)
        if device.type == 'cuda': model.cuda()
        output = model.predict(texts, show_progress_bar=False)
        for sent_preds in output:
            predictions.append({(ent['span'], ent['label']) for ent in sent_preds})
        del model
    elif model_type == "GLINER":
        model = GLiNER.from_pretrained(model_path)
        model.to(device)
        from dataset_processing import CLIRENER_LABELS_V1
        labels = list(CLIRENER_LABELS_V1)
        for text in texts:
            out = model.predict_entities(text, labels, threshold=0.5)
            predictions.append({(ent['text'], ent['label']) for ent in out})
        del model
    torch.cuda.empty_cache()
    return predictions

def get_seed_frequencies(base_dir, model_type, model_id, train_dataset, texts, device, seeds):
    print(f"\n>>> Analyzing Model: {model_id}")
    freq_maps = [Counter() for _ in texts]
    for seed in seeds:
        path = get_output_dir(base_dir, model_type, model_id, train_dataset, seed) / "checkpoint-final"
        if not path.exists(): continue
        print(f"  - Processing Seed {seed}...")
        seed_preds = run_inference_single_seed(str(path), model_type, texts, device)
        for idx, sent_set in enumerate(seed_preds):
            freq_maps[idx].update(sent_set)
        gc.collect()
    return freq_maps

def format_context(text, entity_text):
    start = text.find(entity_text)
    if start == -1: return f"**{entity_text}**"
    end = start + len(entity_text)
    ctx_start, ctx_end = max(0, start - 35), min(len(text), end + 35)
    snippet = text[ctx_start:ctx_end]
    formatted = snippet.replace(entity_text, f"**{entity_text}**")
    return f"{'...' if ctx_start > 0 else ''}{formatted}{'...' if ctx_end < len(text) else ''}"




BASELINE = "FacebookAI/roberta-base"
CHALLENGER = "nasa-impact/nasa-smd-ibm-v0.1"

BASE_T = "SPANMARKER"
CHAL_T = "SPANMARKER"

TRAIN_DATASET = "P0L3/CliReNER_v_1_1_28_SILVER"
EVAL_DATASET = "P0L3/CliReNER_v_1_1_28_GOLD_authorannots"


# 1. Load Data
dataset, id2label = load_gold_data(EVAL_DATASET)
texts, gold_sets = dataset['text'], [extract_gold_spans(row, id2label) for row in dataset]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



# 2. Get frequencies
base_freqs = get_seed_frequencies("EXPERIMENTS/models", BASE_T, BASELINE, TRAIN_DATASET, texts, device, DEFAULT_SEEDS)
chall_freqs = get_seed_frequencies("EXPERIMENTS/models", CHAL_T, CHALLENGER, TRAIN_DATASET, texts, device, DEFAULT_SEEDS)

# 3. Categorization
results = {tag: {"lost": [], "shared": [], "gained": []} for tag in TARGET_TAGS}

print(f"\n--- Categorizing (4-1 Rule: Success >={SUCCESS_THRESHOLD}, Failure <={FAILURE_THRESHOLD}) ---")
    
for i, text in enumerate(texts):
    G = gold_sets[i]
    B_map, C_map = base_freqs[i], chall_freqs[i]

    for (ent_text, ent_label) in G:
        if ent_label not in TARGET_TAGS: continue
        
        b_votes = B_map.get((ent_text, ent_label), 0)
        c_votes = C_map.get((ent_text, ent_label), 0)
        
        ctx = format_context(text, ent_text)

        if b_votes >= SUCCESS_THRESHOLD and c_votes >= SUCCESS_THRESHOLD:
            results[ent_label]["shared"].append(ctx)
        elif b_votes >= SUCCESS_THRESHOLD and c_votes <= FAILURE_THRESHOLD:
            results[ent_label]["lost"].append(ctx)
        elif c_votes >= SUCCESS_THRESHOLD and b_votes <= FAILURE_THRESHOLD:
            results[ent_label]["gained"].append(ctx)



# with open(output_path, "w", encoding="utf-8") as f:
#     f.write(f"# Robust Architectural Comparison: {BASELINE} vs {CHALLENGER}\n")
#     f.write(f"- **Strictness (4-1 Rule):**\n")
#     f.write(f"  - **Stable Success:** Found in $\ge {SUCCESS_THRESHOLD}/5$ seeds.\n")
#     f.write(f"  - **Stable Failure:** Found in $\le {FAILURE_THRESHOLD}/5$ seeds.\n\n")
    
#     f.write(f"| Entity Type | Correct in Baseline ONLY ({base_short}) | Correct in BOTH | Correct in Challenger ONLY ({chall_short}) |\n")
#     f.write("|---|---|---|---|\n")
    
#     for tag in TARGET_TAGS:
#         def sample_cell(key):
#             items = list(set(results[tag][key]))
#             if not items: return "*(None found matching 4-1 rule)*"
#             return "<br><br>".join(random.sample(items, min(2, len(items))))
        
#         f.write(f"| **{tag}** | {sample_cell('lost')} | {sample_cell('shared')} | {sample_cell('gained')} |\n")

print("Done.")



--- Loading Dataset: P0L3/CliReNER_v_1_1_28_GOLD_authorannots ---

>>> Analyzing Model: FacebookAI/roberta-base
  - Processing Seed 0...
  - Processing Seed 42...
  - Processing Seed 3012...
  - Processing Seed 33...
  - Processing Seed 131...

>>> Analyzing Model: nasa-impact/nasa-smd-ibm-v0.1
  - Processing Seed 0...
  - Processing Seed 42...
  - Processing Seed 3012...
  - Processing Seed 33...
  - Processing Seed 131...

--- Categorizing (4-1 Rule: Success >=3, Failure <=2) ---
Done.


In [41]:
for i,text in enumerate(texts):
    if "vegetation growth" in text:
        print(i, text)

83 Unlike relative humidity (RH), VPD provides a linear measure of the exponential relationship between temperature and evapotranspiration (Anderson, 1936), and increasing VPD around the globe has been linked to reduced vegetation growth (Yuan et al., 2019) and wildfires (Abatzoglou et al., 2018b).


In [42]:
chall_freqs[83]

Counter({('VPD', 'Quantity'): 5,
         ('globe', 'Location'): 5,
         ('RH', 'Quantity'): 5,
         ('evapotranspiration', 'Quantity'): 5,
         ('temperature', 'Quantity'): 5,
         ('2018b', 'Time Period'): 5,
         ('relative humidity', 'Quantity'): 5,
         ('2019', 'Time Period'): 5,
         ('linear measure', 'Quantity'): 4,
         ('wildfires', 'Natural Disaster'): 3,
         ('exponential relationship', 'Physical Phenomenon'): 2,
         ('wildfires', 'Meteorological Phenomenon'): 2,
         ('exponential relationship', 'Quantity'): 2,
         ('Abatzoglou et al.,', 'Person'): 1,
         ('exponential relationship', 'Other'): 1,
         ('linear measure', 'Other'): 1,
         ('vegetation growth', 'Organism'): 1,
         ('1936', 'Time Period'): 1,
         ('vegetation growth', 'Natural Phenomenon'): 1,
         ('vegetation growth', 'Meteorological Phenomenon'): 1,
         ('increasing', 'Quantity'): 1,
         ('vegetation growth', 'Physical

In [35]:
results["Natural Phenomenon"]

{'lost': ['...s for acclimation and facilitating **compensatory growth** following drought.',
  '...ay a role in drought tolerance and **drought avoidance** by supplying N to leaves for accli...',
  '...en (N) fixation may play a role in **drought tolerance** and drought avoidance by supplying...',
  '...es linking climate variability and **disease transmission** across the various agro-ecological...',
  '...e globe has been linked to reduced **vegetation growth** (Yuan et al.,\xa02019) and wildfires ...',
  'Species-level analysis of **vegetation patterns** enables quantifying and monitoring...',
  'Therefore, when modeling **vegetation patterns**, ecologists must carefully select ...',
  '...information required to assess the **triggering phenomena**.',
  '...sult in the rapid growth of algae, **eutrophication**, and the death of animal life due ...'],
 'shared': [],
 'gained': []}

--- Loading and Merging GOLD Dataset: P0L3/CliReNER_v_1_1_28_GOLD_authorannots ---
Merged ['train', 'validation', 'test'] into a single dataset of size: 192
Error: File RESULTS/LLM_PREDICTIONS/ner_results_gemini_2_5_pro.jsonl not found.
Aligned 0 rows. (Skipped: 192 missing, 0 length mismatch)
No errors found for entity type: Natural Phenomenon

--- Overall Performance (Strict Match) ---


TypeError: tuple indices must be integers or slices, not str